In [1]:
import pandas as pd
import numpy as np
import sklearn
import torch
print("ok")

ok


In [8]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/public_test.csv")

print(train.shape)
print(test.shape)

print(train["label"].value_counts())
print(test["label"].value_counts())
    
print(train.groupby(["label", "label_name"]).size())

(240, 5)
(400, 5)
label
1    180
0     60
Name: count, dtype: int64
label
1    200
0    200
Name: count, dtype: int64
label  label_name
0      negative       60
1      positive      180
dtype: int64


In [5]:
train.head()

,id,text,label,label_name,source_file
0,pos_cv230_7428,"well , i'll admit when i first heard about thi...",1,positive,pos/cv230_7428.txt
1,pos_cv853_29233,my summer was recently saved by two very diffe...,1,positive,pos/cv853_29233.txt
2,pos_cv771_28665,in october of 1962 the united states found its...,1,positive,pos/cv771_28665.txt
3,pos_cv449_8785,this is a good year if you want plenty of sci-...,1,positive,pos/cv449_8785.txt
4,pos_cv130_17083,"while watching wes anderson's rushmore , it ma...",1,positive,pos/cv130_17083.txt


In [9]:
print(len(set(train["source_file"]) & set(test["source_file"])))
print(train["text"].duplicated().sum())
print(train["text"].str.split().str.len().describe())

0
0
count     240.000000
mean      787.708333
std       364.790842
min        17.000000
25%       555.250000
50%       732.500000
75%       939.750000
max      2570.000000
Name: text, dtype: float64


Data Inspection
Cell 1:
Cell 2:
Cell 3:

In [11]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

def evaluate_model(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    print(f"Accuracy: {acc:.4f}")
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    return acc, bal_acc, cm

In [12]:
train_baseline = np.ones(len(train), dtype=int)
test_baseline = np.ones(len(test), dtype=int)


print("All positive baseline on train set:")
evaluate_model(train["label"], train_baseline)

print("\nAll positive baseline on test set:")
evaluate_model(test["label"], test_baseline)

All positive baseline on train set:
Accuracy: 0.7500
Balanced Accuracy: 0.5000
Confusion Matrix:
[[  0  60]
 [  0 180]]

All positive baseline on test set:
Accuracy: 0.5000
Balanced Accuracy: 0.5000
Confusion Matrix:
[[  0 200]
 [  0 200]]


(0.5,
 0.5,
 array([[  0, 200],
        [  0, 200]]))

In [13]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

token_counts = pd.Series([len(tokenizer(t)["input_ids"]) for t in train["text"]])

print(token_counts.describe())
print(f"\n Reviews with over 512 tokens: {(token_counts > 512).sum()} out of {len(token_counts)}")


c:\Users\dleon\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\dleon\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dleon\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an a

count     240.000000
mean      896.933333
std       409.528022
min        27.000000
25%       635.500000
50%       852.500000
75%      1078.000000
max      2848.000000
dtype: float64

 Reviews with over 512 tokens: 210 out of 240


In [ ]:
def chunk_text(text, max_length=512, overlap=0):
    token_id = tokenizer(text, add_special_tokens=False)["input_ids"]
    body_size = max_length - 2  # account for [CLS] and [SEP] tokens
    step = body_size - overlap
    chunks = []
    for i in range(0, len(token_id), step):
        body = token_id[i:i + body_size]
        chunks.append([tokenizer.cls_token_id] + body + [tokenizer.sep_token_id])
        if i + body_size >= len(token_id):
            break
    return chunks


longest = train["text"].iloc[token_counts.idxmax()]
c = chunk_text(longest)
print(f"chunks: {len(c)}, sizes: {[len(x) for x in c]}")
print(f"total body tokens: {sum(len(x) - 2 for x in c)} (expected {token_counts.max() - 2})")

c2 = chunk_text(longest, overlap=100)
print(f"overlap=100 -> chunks: {len(c2)}, sizes: {[len(x) for x in c2]}")

chunks: 6, sizes: [512, 512, 512, 512, 512, 298]
total body tokens: 2846 (expected 2846)
overlap=100 -> chunks: 7, sizes: [512, 512, 512, 512, 512, 512, 388]


In [21]:
#Frozen econder
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained("distilbert-base-uncased")
model.eval()

print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7342.07it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


parameters: 66,362,880


In [23]:
@torch.no_grad()
def get_embeddings(text, overlap=0):
    chunks =  chunk_text(text, overlap=overlap)
    vectors = []
    for c in chunks:
        input_ids = torch.tensor([c])
        outputs = model(input_ids)
        vectors.append(outputs.last_hidden_state[:, 0, :].squeeze(0))
    return torch.stack(vectors).mean(dim=0).numpy()

v = get_embeddings(train["text"].iloc[0])
print(v.shape)

v_long = get_embeddings(longest)
print(v_long.shape)

(768,)
(768,)
